# Mandarin Tone Discrimination Task Analysis

In [ ]:
import importlib
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
from tqdm.auto import tqdm

import sys
sys.path.append('../')
sys.path.append('../lightning_scripts')
sys.path.append('../model_configs')
sys.path.append('../byol-a')


from lightning_scripts.utils.model_build_utils import get_model
from lightning_scripts.byola_lightning_module import BYOLAModule
from lightning_scripts.tone_perfect_triplet_dataset import TonePerfectDataset

import figure_utils
from importlib import reload
reload(figure_utils)
from figure_utils import (
    normalize_model_name,
    build_model_palette,
    get_standard_hue_order,
    get_standard_base_colors,
    model_label,
    plot_grouped_bars,
)

torch.set_float32_matmul_precision("medium")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

%matplotlib inline

## Dataset Setup

In [ ]:
model_sr = 20_000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Helper Functions

In [ ]:
from lightning_scripts.zero_shot_utils import encode_audio, distance_metrics_on_triplet


## Load Models

In [ ]:
from lightning_scripts.zero_shot_utils import get_cochdnn9_models

eval_layer = "relu4"
models, model_name_map = get_cochdnn9_models(layer_out=eval_layer, device=DEVICE)


## Run Triplet Evaluation

In [ ]:
# Instantiate dataset with balanced triplets
n_examples = 2000  # Total examples, will be divided evenly across tones

tone_perfect_dir = Path("${COCHDNN_TONE_PERFECT_DIR}").expanduser()
tp_dataset = TonePerfectDataset(
    tone_perfect_dir,
    resample_sr=model_sr,
    pair_mode=True,  # two positives: same speaker/tone, different base syllables
    include_negative=True,  # negative: same speaker, different tone
    allow_tone1=True,
    n_examples=n_examples,  # Pre-generate balanced triplets
    random_seed=0,
)

print(f"Dataset ready: {len(tp_dataset)} triplets at target_sr={tp_dataset.resample_sr}")

# Optional: Use DataLoader like NSynth notebook
def mandarin_triplet_collate(batch):
    clips, sr, triplet = batch[0]
    return clips, sr, triplet

tp_loader = torch.utils.data.DataLoader(
    tp_dataset, batch_size=1, shuffle=False, collate_fn=mandarin_triplet_collate
)

# Simple evaluation loop - iterate through pre-generated triplets

records = []
for batch_idx, (clips, sr, triplet) in tqdm(enumerate(tp_loader), total=len(tp_loader), desc="Evaluating triplets"):
    sr_val = int(sr)
    
    # Skip if any audio is None (shouldn't happen with pre-generated triplets, but check anyway)
    if clips["anchor"] is None or clips["positive"] is None or clips["negative"] is None:
        continue

    for internal_name, model in models.items():
        metrics = distance_metrics_on_triplet(model, clips, sr_val, device=DEVICE, include_cosine=True)

        # Standardize model naming for downstream plotting
        pretty_name = model_name_map.get(internal_name, internal_name)
        metrics.update({
             "example_idx": batch_idx,
            "model": internal_name,
            "model_name": normalize_model_name(pretty_name),
            "speaker": triplet["anchor"].get("speaker", None),
            "tone": triplet["anchor"].get("tone", None),
            "syllable_a": triplet["anchor"].get("syllable", None),
            "syllable_b": triplet["positive"].get("syllable", None),
            "neg_tone": triplet["negative"].get("tone", None),
        })
        records.append(metrics)

mandarin_tone_df = pd.DataFrame(records)
print(
    f"Evaluated {mandarin_tone_df.example_idx.nunique()} triplets; "
    f"{len(mandarin_tone_df)} model-evals"
)
mandarin_tone_df.head()

In [ ]:
## write out results to csv

out_dir = Path("../results_dfs")
out_dir.mkdir(exist_ok=True, parents=True)
mandarin_tone_df.to_csv(out_dir / f"zero_shot_mandarin_tone_results_{eval_layer}_v2.csv", index=False)


In [ ]:
mandarin_tone_df.model_name.unique()

In [ ]:
# Get standardized hue order and colors
hue_order = get_standard_hue_order()
base_colors = get_standard_base_colors()
hue_dict = build_model_palette(hue_order, base_colors)

# Use plot_grouped_bars for the main squared L2 norm plot
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

_, legend_handles, legend_labels = plot_grouped_bars(
    ax,
    mandarin_tone_df,
    value_col='r_judgement',
    title=f'Mandarin tone discrimination {eval_layer}',
    ylabel='Prop. hit rate',
    error_type='sem',
    hue_order=hue_order,
    hue_dict=hue_dict,
    capsize=0,
)

ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1)
ax.grid(axis='y', alpha=0.3, zorder=-1)

# Add legend
if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=4,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Get standardized hue order and colors
hue_order = get_standard_hue_order()
base_colors = get_standard_base_colors()
hue_dict = build_model_palette(hue_order, base_colors)

# Use plot_grouped_bars for the main squared L2 norm plot
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

_, legend_handles, legend_labels = plot_grouped_bars(
    ax,
    mandarin_tone_df,
    value_col='cos_judgement',
    title=f'Mandarin tone discrimination {eval_layer}',
    ylabel='Prop. hit rate',
    error_type='sem',
    hue_order=hue_order,
    hue_dict=hue_dict,
    capsize=0,
)

ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1)
ax.grid(axis='y', alpha=0.3, zorder=-1)

# Add legend
if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=4,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Get standardized hue order and colors
hue_order = get_standard_hue_order()
base_colors = get_standard_base_colors()
hue_dict = build_model_palette(hue_order, base_colors)

# Use plot_grouped_bars for the main squared L2 norm plot
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

_, legend_handles, legend_labels = plot_grouped_bars(
    ax,
    mandarin_tone_df,
    value_col='sqr_l2_judgement',
    title=f'Mandarin tone discrimination {eval_layer}',
    ylabel='Prop. hit rate',
    error_type='sem',
    hue_order=hue_order,
    hue_dict=hue_dict,
    capsize=0,
)

ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1)
ax.grid(axis='y', alpha=0.3, zorder=-1)

# Add legend
if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=4,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Get standardized hue order and colors
hue_order = get_standard_hue_order()
base_colors = get_standard_base_colors()
hue_dict = build_model_palette(hue_order, base_colors)

# Use plot_grouped_bars for the main squared L2 norm plot
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

legend_handles, legend_labels = plot_grouped_bars(
    ax,
    mandarin_tone_df,
    value_col='sqr_l2_judgement',
    title='Mandarin tone discrimination',
    ylabel='Prop. hit rate',
    error_type='sem',
    hue_order=hue_order,
    hue_dict=hue_dict,
    capsize=0,
)

ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1)
ax.grid(axis='y', alpha=0.3, zorder=-1)

# Add legend
if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=4,
    )

plt.tight_layout()
plt.show()

## Plot Results - All Metrics Comparison

In [ ]:
# For comparison across metrics, use errorbar plots with figure_utils styling
fig, axes = plt.subplots(1, 2, figsize=(10, 6), constrained_layout=False)

# Filter to only models we have data for
available_models = mandarin_tone_df['model_name'].unique()
hue_order_filtered = [m for m in hue_order if m in available_models]

# plot mean judgement for each model
for ix, metric in enumerate(["sqr_l2_judgement", "cos_judgement"]):
    for model in hue_order_filtered:
        model_data = mandarin_tone_df[mandarin_tone_df.model_name == model][metric]
        if len(model_data) == 0:
            continue
        mean_val = model_data.mean()
        error = 2 * model_data.std() / np.sqrt(len(model_data))
        
        x_pos = hue_order_filtered.index(model)
        color = hue_dict.get(model, '#444444')
        
        axes[ix].errorbar(
            x_pos,
            mean_val,
            fmt='o',
            yerr=error,
            color=color,
            markeredgecolor="k",
            markersize=8,
            capsize=4,
        )
    
    axes[ix].set_title(f"Mandarin tone discrimination\n{metric}")
    axes[ix].set_ylabel("Prop. hit rate")
    axes[ix].set_xlabel("Model")
    axes[ix].grid(True, alpha=0.3)
    axes[ix].set_ylim(0.5, 1)
    axes[ix].set_xticks(range(len(hue_order_filtered)))
    axes[ix].set_xticklabels([model_label(m) for m in hue_order_filtered], rotation=90, ha='center')

plt.tight_layout()
plt.show()

## Load ResNet18 models from eval_dict and supervised models

In [ ]:
import pickle

# Load ResNet18 SSL models from eval_dict
eval_dict_path = Path("train_config_manifests/resnet18_barlow_equivariant_lmbda_search_eval_dict.pkl")
eval_dict = pickle.load(open(eval_dict_path, 'rb'))

# ResNet18 supervised models
resnet18_word_config = Path("model_configs/supervised_models/word_resnet18_MatchedDataset_LARS.yaml")
resnet18_audioset_config = Path("model_configs/supervised_models/audioset_resnet18_MatchedDataset_LARS.yaml")
resnet18_multitask_config = Path("model_configs/supervised_models/word_speaker_audioset_resnet18_MatchedDataset_shuffle_one_gpu_LARS.yaml")

# Load ResNet18 SSL models from eval_dict
resnet18_models = {}
resnet18_model_name_map = {}

for idx, config_path in eval_dict.items():
    config_path = Path(config_path)
    # Extract lambda value from config name for naming
    config_name = config_path.stem
    if 'eq_lmbda' in config_name:
        # Extract lambda value (format: eq_lmbda_5e-01 means 0.5)
        import re
        match = re.search(r'eq_lmbda_(\d+)e-(\d+)', config_name)
        if match:
            coeff = float(match.group(1))
            exp = float(match.group(2))
            lambda_val = coeff * (10 ** -exp)
            model_key = f"resnet18_ssl_λ={lambda_val}"
            resnet18_models[model_key] = get_model(config_path, layer_out="layer4")
            resnet18_model_name_map[model_key] = f"ResNet18 ssl λ={lambda_val}"
        else:
            # Fallback naming
            model_key = f"resnet18_ssl_{idx}"
            resnet18_models[model_key] = get_model(config_path, layer_out="layer4")
            resnet18_model_name_map[model_key] = f"ResNet18 ssl {config_name}"
    elif 'invariant_only' in config_name:
        model_key = "resnet18_ssl_invar"
        resnet18_models[model_key] = get_model(config_path, layer_out="layer4")
        resnet18_model_name_map[model_key] = "ResNet18 ssl invar"

# Load ResNet18 supervised models
resnet18_models["resnet18_word"] = get_model(resnet18_word_config, supervised=True, layer_out="layer4")
resnet18_model_name_map["resnet18_word"] = "ResNet18 supervised word"

resnet18_models["resnet18_audioset"] = get_model(resnet18_audioset_config, supervised=True, layer_out="layer4")
resnet18_model_name_map["resnet18_audioset"] = "ResNet18 supervised audioset"

resnet18_models["resnet18_multitask"] = get_model(resnet18_multitask_config, supervised=True, layer_out="layer4")
resnet18_model_name_map["resnet18_multitask"] = "ResNet18 supervised multi-task"

# Push all ResNet18 models to device
for model in resnet18_models.values():
    model.to(DEVICE)

print(f"Loaded {len(resnet18_models)} ResNet18 models")

## Evaluate ResNet18 models

In [ ]:
# Evaluate ResNet18 models on triplets
resnet18_records = []
for batch_idx, (clips, sr, triplet) in tqdm(enumerate(tp_loader), total=len(tp_loader), desc="Evaluating ResNet18 triplets"):
    sr_val = int(sr)
    
    # Skip if any audio is None
    if clips["anchor"] is None or clips["positive"] is None or clips["negative"] is None:
        continue

    for internal_name, model in resnet18_models.items():
        metrics = distance_metrics_on_triplet(model, clips, sr_val, device=DEVICE, include_cosine=True)

        # Standardize model naming for downstream plotting
        pretty_name = resnet18_model_name_map.get(internal_name, internal_name)

        resnet18_records.append(
            {
                "example_idx": batch_idx,
                "model": internal_name,
                "model_name": normalize_model_name(pretty_name),
                "speaker": triplet["anchor"].get("speaker", None),
                "tone": triplet["anchor"].get("tone", None),
                "syllable_a": triplet["anchor"].get("syllable", None),
                "syllable_b": triplet["positive"].get("syllable", None),
                "neg_tone": triplet["negative"].get("tone", None),
                # metrics used by plotting below
                "l2_judgement": metrics["l2_judgement"],
                "sqr_l2_judgement": metrics["sqr_l2_judgement"],
                "cos_judgement": metrics["cos_judgement"],
                # optional diagnostics
                "pos_l2": metrics["pos_l2"],
                "neg_l2": metrics["neg_l2"],
                "pos_cos": metrics["pos_cos"],
                "neg_cos": metrics["neg_cos"],
            }
        )

resnet18_mandarin_tone_df = pd.DataFrame(resnet18_records)
print(
    f"Evaluated {resnet18_mandarin_tone_df.example_idx.nunique()} triplets; "
    f"{len(resnet18_mandarin_tone_df)} model-evals"
)
resnet18_mandarin_tone_df.head()

## Plot ResNet18 results

In [ ]:
# Get standardized hue order and colors
hue_order = get_standard_hue_order()
base_colors = get_standard_base_colors()
hue_dict = build_model_palette(hue_order, base_colors)

# Use plot_grouped_bars for the main squared L2 norm plot
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

legend_handles, legend_labels = plot_grouped_bars(
    ax,
    resnet18_mandarin_tone_df,
    value_col='sqr_l2_judgement',
    title='Mandarin tone discrimination (ResNet18)',
    ylabel='Prop. hit rate',
    hue_order=hue_order,
    hue_dict=hue_dict,
)

ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1)
ax.grid(axis='y', alpha=0.3, zorder=-1)

# Add legend
if legend_handles:
    fig.legend(
        legend_handles,
        legend_labels,
        frameon=True,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=4,
    )

plt.tight_layout()
plt.show()